In [48]:
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [50]:
loader = PyPDFLoader("/Users/poojaveluri/Documents/genai/data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [51]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

26

In [52]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")

In [53]:
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding= embeddings,
    persist_directory='./vector_agentic_rag_db'
)

## agent = tools + llm + prompt

In [54]:
@tool
def retriever_tool(query:str):
    """
    This tool helps to retreive the relavant data from PDF document. and these pdf has details about medical reports
    """
    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""
    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [55]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash-lite")

In [56]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the retriever_tool tool for questions requiring external knowledge.
    """

In [57]:
agent = create_agent(
    model = llm,
    tools = [retriever_tool],
    system_prompt=System_Prompt
)

In [62]:
query = "tell me about vitamin D in the medical report"
response = agent.invoke({"messages":[{"role":"user","content":query}]})
result = response["messages"][-1].text
print(result)

Tool Called:  vitamin D
Tool Called:  VITAMIN D interpretation comments range
Based on the medical report:

* **Test Name:** Vitamin D, 25 - Hydroxy, Serum (ECLIA)
* **Result:** 67.50 nmol/L
* **Biological Reference Interval:** 75.00 - 250.00 nmol/L

**Interpretation:** 
Your Vitamin D level is **67.50 nmol/L**, which is slightly below the standard reference interval range of 75.00 to 250.00 nmol/L, indicating a mild deficiency or suboptimal level. It is recommended to consult your physician to discuss these results and determine if any supplementation or lifestyle changes are necessary.
